In [ ]:
import numpy as np 
import pandas as pd 
import os
import kagglehub
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

path = kagglehub.competition_download('ml-opsidian-genesis-initial-round-26')
train_path = os.path.join(path, 'train.csv')
test_path = os.path.join(path, 'test.csv')

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

num_features = [
    'latitude', 'longitude', 'elevation_m', 'distance_to_river_m',
    'population_density_per_km2', 'built_up_percent', 'rainfall_7d_mm',
    'monthly_rainfall_mm', 'drainage_index', 'ndvi', 'ndwi',
    'historical_flood_count', 'infrastructure_score', 'nearest_hospital_km',
    'nearest_evac_km', 'seasonal_index', 'terrain_roughness_index',
    'socioeconomic_status_index', 'extreme_weather_index'
]

# Grouping categorical labels
cat_features = [
    'district', 'landcover', 'soil_type', 'water_supply',
    'electricity', 'road_quality', 'urban_rural',
    'water_presence_flag', 'flood_occurrence_current_event'
]

# Note: We are dropping the pre-transformed (log/yeojohnson) columns here to 
# let our pipeline handle the raw features uniformly, keeping the model simpler.
cols_to_drop = [
    'record_id', 'place_name', 'reason_not_good_to_live', 
    'is_good_to_live', 'is_synthetic', 'generation_date', 'inundation_area_sqm'
]

# 3. Separate Features and Target
print("Setting up training sets...")
X = df_train.drop(columns=['flood_risk_score'] + cols_to_drop + 
                  [c for c in df_train.columns if '_log1p' in c or '_yeojohnson' in c or '_qmap' in c])
y = df_train['flood_risk_score']

# Ensure the test set perfectly matches the training set columns
X_test = df_test[X.columns]

# 4. Build the MLOps Pipeline with Imputers
# Impute with median for numericals to resist outliers, then scale
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Impute with a constant for categoricals, then One-Hot Encode
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

# 5. Create the full modeling pipeline
# n_jobs=-1 tells the model to use all available CPU cores for faster training
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=150, max_depth=15, random_state=42, n_jobs=-1)) 
])

# 6. Train and Predict
print("Training model (this might take a minute)...")
model_pipeline.fit(X, y)

print("Generating predictions...")
predictions = model_pipeline.predict(X_test)

# The target variable must strictly be between 0 and 1
predictions = np.clip(predictions, 0, 1)

submission = pd.DataFrame({
    'record_id': df_test['record_id'],
    'flood_risk_score': predictions
})

submission.to_csv('submission.csv', index=False)
print("Success! submission.csv saved.")

ModuleNotFoundError: No module named 'pandas'